# LightGBM Tuning with TRAIN-Only Temporal Cross-Validation

The external validation period has already guided feature experiments, so hyperparameters are selected only through expanding-window CV inside TRAIN. Candidate A uses frequency plus historical amounts; Candidate B adds the existing eight graph features. The external validation period is evaluated once per selected candidate, and the reserved final split is never accessed.

In [1]:
import gc
import json
import sys
import time
from pathlib import Path

import lightgbm as lgb
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.data import (
    load_train_identity,
    load_train_transaction,
    merge_train_data,
    split_data_chronologically,
)
from fraud_detection.models import (
    evaluate_lightgbm_fold,
    evaluate_probabilities,
    make_expanding_time_folds,
    prepare_engineered_matrices,
    summarize_lightgbm_cv,
)

notebook_started = time.perf_counter()

## Why temporal CV

Random folds would train on future fraud patterns. Three expanding folds instead use approximately 70%→10%, 80%→10%, and 90%→10% of TRAIN. Boundaries move to the start of a complete `TransactionDT`, guaranteeing every fit timestamp is strictly earlier than its evaluation period. Each fold independently fits frequency encoding, preprocessing, historical amount state, and—when selected—graph state.

In [2]:
raw_dir = PROJECT_ROOT / "data" / "raw"
transaction = load_train_transaction(raw_dir)
identity = load_train_identity(raw_dir)
merged = merge_train_data(transaction, identity)
split = split_data_chronologically(merged)
train = split.train
validation = split.validation
del transaction, identity, merged
gc.collect()

folds = make_expanding_time_folds(train, n_splits=3, initial_train_fraction=0.70)
fold_rows = []
for fold_number, fold in enumerate(folds, start=1):
    fold_train = train.iloc[fold.train_indices]
    fold_evaluation = train.iloc[fold.evaluation_indices]
    fold_rows.append(
        {
            "Fold": fold_number,
            "Fit Rows": len(fold_train),
            "Evaluation Rows": len(fold_evaluation),
            "Fit DT Max": fold_train["TransactionDT"].max(),
            "Evaluation DT Min": fold_evaluation["TransactionDT"].min(),
            "Evaluation DT Max": fold_evaluation["TransactionDT"].max(),
        }
    )
fold_design = pd.DataFrame(fold_rows)
display(fold_design)
print("External validation status: reserved until CV selection is complete")
print("Final holdout status: not assigned or accessed")

,Fold,Fit Rows,Evaluation Rows,Fit DT Max,Evaluation DT Min,Evaluation DT Max
0,1,289364,41338,7135483,7135609,8121450
1,2,330702,41338,8121450,8121466,9247568
2,3,372040,41338,9247568,9247583,10437996


External validation status: reserved until CV selection is complete
Final holdout status: not assigned or accessed


In [3]:
common_parameters = {
    "objective": "binary",
    "metric": "average_precision",
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}
candidate_parameters = [
    dict(
        learning_rate=0.10,
        n_estimators=500,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=20,
        colsample_bytree=1.0,
        subsample=1.0,
        subsample_freq=0,
        reg_alpha=0.0,
        reg_lambda=0.0,
    ),
    dict(
        learning_rate=0.05,
        n_estimators=800,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=30,
        colsample_bytree=0.9,
        subsample=0.9,
        subsample_freq=1,
        reg_alpha=0.1,
        reg_lambda=0.5,
    ),
    dict(
        learning_rate=0.05,
        n_estimators=800,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=30,
        colsample_bytree=0.9,
        subsample=0.9,
        subsample_freq=1,
        reg_alpha=0.1,
        reg_lambda=0.5,
    ),
    dict(
        learning_rate=0.03,
        n_estimators=1200,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=50,
        colsample_bytree=0.8,
        subsample=0.8,
        subsample_freq=1,
        reg_alpha=0.5,
        reg_lambda=1.0,
    ),
    dict(
        learning_rate=0.03,
        n_estimators=1000,
        num_leaves=127,
        max_depth=8,
        min_child_samples=40,
        colsample_bytree=0.8,
        subsample=0.8,
        subsample_freq=1,
        reg_alpha=0.5,
        reg_lambda=1.0,
    ),
    dict(
        learning_rate=0.03,
        n_estimators=1200,
        num_leaves=63,
        max_depth=8,
        min_child_samples=50,
        colsample_bytree=0.9,
        subsample=0.8,
        subsample_freq=1,
        reg_alpha=1.0,
        reg_lambda=2.0,
    ),
    dict(
        learning_rate=0.10,
        n_estimators=500,
        num_leaves=31,
        max_depth=6,
        min_child_samples=40,
        colsample_bytree=0.8,
        subsample=1.0,
        subsample_freq=0,
        reg_alpha=0.0,
        reg_lambda=1.0,
    ),
    dict(
        learning_rate=0.07,
        n_estimators=700,
        num_leaves=63,
        max_depth=8,
        min_child_samples=30,
        colsample_bytree=0.85,
        subsample=0.9,
        subsample_freq=1,
        reg_alpha=0.1,
        reg_lambda=1.0,
    ),
]
candidate_parameters = [{**common_parameters, **parameters} for parameters in candidate_parameters]
parameter_table = pd.DataFrame(candidate_parameters)
parameter_table.index = np.arange(1, len(parameter_table) + 1)
parameter_table.index.name = "Configuration"
display(parameter_table.drop(columns=["objective", "metric", "verbosity", "n_jobs"]))

,class_weight,random_state,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,colsample_bytree,subsample,subsample_freq,reg_alpha,reg_lambda
Configuration,,,,,,,,,,,,
1,balanced,42,0.10,500,63,-1,20,1.00,1.0,0,0.0,0.0
2,balanced,42,0.05,800,31,-1,30,0.90,0.9,1,0.1,0.5
3,balanced,42,0.05,800,63,-1,30,0.90,0.9,1,0.1,0.5
4,balanced,42,0.03,1200,31,-1,50,0.80,0.8,1,0.5,1.0
5,balanced,42,0.03,1000,127,8,40,0.80,0.8,1,0.5,1.0
6,balanced,42,0.03,1200,63,8,50,0.90,0.8,1,1.0,2.0
7,balanced,42,0.10,500,31,6,40,0.80,1.0,0,0.0,1.0
8,balanced,42,0.07,700,63,8,30,0.85,0.9,1,0.1,1.0


In [4]:
tracking_uri = f"sqlite:///{(PROJECT_ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-lightgbm-tuning")
preparation_seconds = {}
maximum_matrix_mb = 0.0


def run_candidate_cv(feature_set, include_graph):
    global maximum_matrix_mb
    fold_evaluations = [[] for _ in candidate_parameters]
    feature_count = 0
    candidate_started = time.perf_counter()
    for fold_number, fold in enumerate(folds, start=1):
        fold_train = train.iloc[fold.train_indices]
        fold_evaluation = train.iloc[fold.evaluation_indices]
        prepared = prepare_engineered_matrices(
            fold_train, fold_evaluation, include_graph=include_graph
        )
        feature_count = prepared.feature_count
        matrix_mb = (prepared.x_train.nbytes + prepared.x_evaluation.nbytes) / 1024**2
        maximum_matrix_mb = max(maximum_matrix_mb, matrix_mb)
        y_fold_train = fold_train["isFraud"].astype("int8")
        y_fold_evaluation = fold_evaluation["isFraud"].astype("int8")
        for configuration, parameters in enumerate(candidate_parameters):
            evaluation = evaluate_lightgbm_fold(
                parameters,
                prepared.x_train,
                y_fold_train,
                prepared.x_evaluation,
                y_fold_evaluation,
                early_stopping_rounds=50,
            )
            fold_evaluations[configuration].append(evaluation)
            print(
                f"{feature_set} fold={fold_number} config={configuration + 1} "
                f"PR-AUC={evaluation.pr_auc:.6f} iteration={evaluation.best_iteration}"
            )
        del prepared, fold_train, fold_evaluation, y_fold_train, y_fold_evaluation
        gc.collect()

    summaries = [
        summarize_lightgbm_cv(parameters, evaluations)
        for parameters, evaluations in zip(candidate_parameters, fold_evaluations, strict=True)
    ]
    rows = []
    for configuration, summary in enumerate(summaries, start=1):
        with mlflow.start_run(run_name=f"{feature_set}-cv-config-{configuration}") as run:
            mlflow.set_tags(
                {
                    "stage": "model-tuning",
                    "dataset": "ieee-cis",
                    "selection": "temporal-cv",
                    "feature_set": feature_set,
                }
            )
            mlflow.log_params(
                {"feature_set": feature_set, "configuration": configuration, **summary.parameters}
            )
            metrics = {
                "mean_cv_pr_auc": summary.mean_pr_auc,
                "std_cv_pr_auc": summary.std_pr_auc,
                "mean_cv_roc_auc": summary.mean_roc_auc,
            }
            for fold_index, (pr_auc, roc_auc, iteration) in enumerate(
                zip(
                    summary.fold_pr_auc,
                    summary.fold_roc_auc,
                    summary.best_iterations,
                    strict=True,
                ),
                start=1,
            ):
                metrics[f"fold_{fold_index}_pr_auc"] = pr_auc
                metrics[f"fold_{fold_index}_roc_auc"] = roc_auc
                metrics[f"fold_{fold_index}_best_iteration"] = iteration
            mlflow.log_metrics(metrics)
            rows.append(
                {
                    "Feature Set": feature_set,
                    "Configuration": configuration,
                    "Fold PR-AUC": summary.fold_pr_auc,
                    "Mean PR-AUC": summary.mean_pr_auc,
                    "Std PR-AUC": summary.std_pr_auc,
                    "Mean ROC-AUC": summary.mean_roc_auc,
                    "Best Iterations": summary.best_iterations,
                    "Feature Count": feature_count,
                    "MLflow Run ID": run.info.run_id,
                }
            )
    preparation_seconds[feature_set] = time.perf_counter() - candidate_started
    results = pd.DataFrame(rows).sort_values("Mean PR-AUC", ascending=False)
    return results.reset_index(drop=True), summaries


candidate_a_cv, candidate_a_summaries = run_candidate_cv("candidate_a", False)
candidate_b_cv, candidate_b_summaries = run_candidate_cv("candidate_b_graph", True)
display(candidate_a_cv)
display(candidate_b_cv)

2026/10/06 13:51:13 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-lightgbm-tuning' does not exist. Creating a new experiment.


candidate_a fold=1 config=1 PR-AUC=0.544507 iteration=192


candidate_a fold=1 config=2 PR-AUC=0.534405 iteration=430


candidate_a fold=1 config=3 PR-AUC=0.565115 iteration=539


candidate_a fold=1 config=4 PR-AUC=0.537472 iteration=701


...

candidate_b_graph fold=3 config=6 PR-AUC=0.617363 iteration=727


candidate_b_graph fold=3 config=7 PR-AUC=0.610775 iteration=368


candidate_b_graph fold=3 config=8 PR-AUC=0.626501 iteration=631


,Feature Set,Configuration,Fold PR-AUC,Mean PR-AUC,Std PR-AUC,Mean ROC-AUC,Best Iterations,Feature Count,MLflow Run ID
0,candidate_a,3,"(0.5651151144534605, 0.5763681339175115, 0.636...",0.592712,0.031409,0.915088,"(539, 563, 800)",443,3a9efa1f48b44b8f9fcd14769843acfe
1,candidate_a,6,"(0.5654997597506407, 0.5649197026562595, 0.628...",0.586372,0.029929,0.916726,"(1104, 920, 1194)",443,8501f91d2a3b4d8d9546d6c839b3e54e
2,candidate_a,5,"(0.5707887647493946, 0.5576968974516661, 0.623...",0.584065,0.028537,0.914372,"(990, 796, 854)",443,50f0e3d418824d07aa0c06f29523e814
3,candidate_a,8,"(0.5498395139177893, 0.5538824235827299, 0.621...",0.574939,0.032679,0.913039,"(210, 287, 475)",443,e49ca688f7c84077a561a51f800196d5
4,candidate_a,4,"(0.5374721735027582, 0.5550070582001249, 0.621...",0.571440,0.036350,0.916148,"(701, 858, 967)",443,ca7f4cb2963e4ced8c93c8d41da60122
5,candidate_a,2,"(0.5344051974756612, 0.553608157384346, 0.6233...",0.570466,0.038231,0.914998,"(430, 665, 800)",443,1a0002ca3a024da0b224ce14aca7674d
6,candidate_a,1,"(0.5445072182294549, 0.545634582657379, 0.6045...",0.564897,0.028042,0.911508,"(192, 142, 113)",443,6e41e01287cd4bf78c76a76a7841aa40
7,candidate_a,7,"(0.5270278798388955, 0.5328774545979569, 0.622...",0.560885,0.043810,0.908583,"(155, 337, 499)",443,0b4572ab9c794f64afd5ff76acaf2f89


,Feature Set,Configuration,Fold PR-AUC,Mean PR-AUC,Std PR-AUC,Mean ROC-AUC,Best Iterations,Feature Count,MLflow Run ID
0,candidate_b_graph,3,"(0.5490362347749769, 0.5686344014228857, 0.636...",0.584700,0.037443,0.917309,"(239, 455, 800)",451,abf85d6a1913406ca39103dc6f478c83
1,candidate_b_graph,1,"(0.5628348022911034, 0.5549634375397133, 0.626...",0.581505,0.032130,0.909486,"(170, 250, 499)",451,697b91b5bb8b402183bbbd33f0c0c543
2,candidate_b_graph,5,"(0.559517673252999, 0.5601411786674015, 0.6224...",0.580718,0.029541,0.914669,"(872, 680, 999)",451,ff82f392c9e74a318b2204c8be45cca3
3,candidate_b_graph,6,"(0.54356863331928, 0.5623279922158412, 0.61736...",0.574420,0.031316,0.917228,"(389, 964, 727)",451,15ae242d1e404524924db0941df06825
4,candidate_b_graph,8,"(0.5481393643920802, 0.5438670146782617, 0.626...",0.572836,0.037987,0.910370,"(480, 366, 631)",451,880535b6cb5f455e856ddbad511c57a8
5,candidate_b_graph,2,"(0.5380685132535602, 0.5444142175159773, 0.619...",0.567260,0.036887,0.914136,"(437, 380, 670)",451,ce2578725b69437299ff23091e661e3a
6,candidate_b_graph,4,"(0.5334558898513013, 0.5537709861594247, 0.605...",0.564247,0.030336,0.915947,"(555, 964, 573)",451,de64dcda50d241c1a1c5d91aed944dc8
7,candidate_b_graph,7,"(0.515264799868806, 0.5248400435315671, 0.6107...",0.550293,0.042945,0.907087,"(202, 282, 368)",451,1a0772c1715e4cb88b23850b450292ee


## Full TRAIN refit and one-time external VALIDATION evaluation

The best configuration for each feature candidate is now fixed by mean TRAIN-CV PR-AUC. Only at this point is the external validation period transformed and scored. Its labels did not participate in preprocessing, early stopping, configuration ranking, or iteration selection.

In [5]:
best_a_row = candidate_a_cv.iloc[0]
best_b_row = candidate_b_cv.iloc[0]
best_a = candidate_a_summaries[int(best_a_row["Configuration"]) - 1]
best_b = candidate_b_summaries[int(best_b_row["Configuration"]) - 1]
final_rows = []
final_metrics_by_candidate = {}


def fit_final_candidate(feature_set, include_graph, cv_result):
    prepared = prepare_engineered_matrices(train, validation, include_graph=include_graph)
    final_parameters = dict(cv_result.parameters)
    final_parameters["n_estimators"] = max(
        1, int(round(float(np.median(cv_result.best_iterations))))
    )
    model = lgb.LGBMClassifier(**final_parameters)
    model.fit(prepared.x_train, train["isFraud"].astype("int8"))
    probabilities = model.predict_proba(prepared.x_evaluation)[:, 1]
    metrics = evaluate_probabilities(validation["isFraud"].astype("int8"), probabilities)
    with mlflow.start_run(run_name=f"{feature_set}-final-refit") as run:
        mlflow.set_tags(
            {
                "stage": "model-tuning",
                "dataset": "ieee-cis",
                "selection": "temporal-cv",
                "feature_set": feature_set,
                "final_refit": "true",
            }
        )
        mlflow.log_params(
            {
                "feature_set": feature_set,
                "feature_count": prepared.feature_count,
                **final_parameters,
            }
        )
        mlflow.log_metrics(
            {
                "external_validation_pr_auc": metrics.pr_auc,
                "external_validation_roc_auc": metrics.roc_auc,
                "external_validation_precision_at_0_50": metrics.precision,
                "external_validation_recall_at_0_50": metrics.recall,
                "external_validation_f1_at_0_50": metrics.f1,
                "feature_count": prepared.feature_count,
            }
        )
        run_id = run.info.run_id
    maximum_matrix = (prepared.x_train.nbytes + prepared.x_evaluation.nbytes) / 1024**2
    del prepared, model, probabilities
    gc.collect()
    return metrics, final_parameters, run_id, maximum_matrix


for feature_set, include_graph, cv_result in (
    ("candidate_a", False, best_a),
    ("candidate_b_graph", True, best_b),
):
    metrics, parameters, run_id, matrix_mb = fit_final_candidate(
        feature_set, include_graph, cv_result
    )
    maximum_matrix_mb = max(maximum_matrix_mb, matrix_mb)
    final_metrics_by_candidate[feature_set] = metrics
    final_rows.append(
        {
            "Feature Set": feature_set,
            "Feature Count": 451 if include_graph else 443,
            "PR-AUC": metrics.pr_auc,
            "ROC-AUC": metrics.roc_auc,
            "Precision": metrics.precision,
            "Recall": metrics.recall,
            "F1": metrics.f1,
            "Confusion Matrix": metrics.confusion_matrix,
            "Parameters": parameters,
            "MLflow Run ID": run_id,
        }
    )
final_results = pd.DataFrame(final_rows).sort_values("PR-AUC", ascending=False)
display(final_results)

,Feature Set,Feature Count,PR-AUC,ROC-AUC,Precision,Recall,F1,Confusion Matrix,Parameters,MLflow Run ID
0,candidate_a,443,0.593452,0.927279,0.470946,0.610125,0.531577,"((83454, 2085), (1186, 1856))","{'objective': 'binary', 'metric': 'average_pre...",7cea2e213e5b40a39622f1fe259f3152
1,candidate_b_graph,451,0.588683,0.926108,0.414598,0.636752,0.502204,"((82804, 2735), (1105, 1937))","{'objective': 'binary', 'metric': 'average_pre...",6da1a5946cf244f2bc68ceeb2011f229


In [6]:
winner = final_results.iloc[0]
simple_result = final_results.loc[final_results["Feature Set"] == "candidate_a"].iloc[0]
graph_result = final_results.loc[final_results["Feature Set"] == "candidate_b_graph"].iloc[0]
improvement_over_previous = winner["PR-AUC"] - 0.594611
graph_gain = graph_result["PR-AUC"] - simple_result["PR-AUC"]
if graph_gain > 0.001:
    tradeoff = "The graph gain exceeds 0.001 PR-AUC and may justify its added state and latency."
elif graph_gain > 0:
    tradeoff = (
        "Graph features rank slightly better, but the simpler 443-feature model may be "
        "preferable operationally because the gain is below 0.001 PR-AUC."
    )
else:
    tradeoff = (
        "Graph features do not improve tuned validation PR-AUC; the simpler 443-feature "
        "model is preferable on both ranking performance and operational complexity."
    )


def cv_markdown(frame):
    rows = [
        "| Feature set | Config | Fold PR-AUC | Mean | Std | Mean ROC-AUC | Iterations |",
        "| --- | ---: | --- | ---: | ---: | ---: | --- |",
    ]
    for _, row in frame.iterrows():
        fold_values = " / ".join(f"{value:.6f}" for value in row["Fold PR-AUC"])
        iterations = " / ".join(str(value) for value in row["Best Iterations"])
        rows.append(
            f"| {row['Feature Set']} | {int(row['Configuration'])} | {fold_values} | "
            f"{row['Mean PR-AUC']:.6f} | {row['Std PR-AUC']:.6f} | "
            f"{row['Mean ROC-AUC']:.6f} | {iterations} |"
        )
    return "\n".join(rows)


def final_markdown(frame):
    rows = [
        "| Candidate | Features | PR-AUC | ROC-AUC | Precision | Recall | F1 | Confusion |",
        "| --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |",
    ]
    for _, row in frame.iterrows():
        rows.append(
            f"| {row['Feature Set']} | {int(row['Feature Count'])} | "
            f"{row['PR-AUC']:.6f} | {row['ROC-AUC']:.6f} | "
            f"{row['Precision']:.6f} | {row['Recall']:.6f} | {row['F1']:.6f} | "
            f"`{row['Confusion Matrix']}` |"
        )
    return "\n".join(rows)


def fold_markdown(frame):
    rows = [
        "| Fold | Fit rows | Evaluation rows | Fit DT max | Evaluation DT min | "
        "Evaluation DT max |",
        "| ---: | ---: | ---: | ---: | ---: | ---: |",
    ]
    for _, row in frame.iterrows():
        rows.append(
            f"| {int(row['Fold'])} | {int(row['Fit Rows'])} | "
            f"{int(row['Evaluation Rows'])} | {row['Fit DT Max']} | "
            f"{row['Evaluation DT Min']} | {row['Evaluation DT Max']} |"
        )
    return "\n".join(rows)


combined_cv = pd.concat([candidate_a_cv, candidate_b_cv], ignore_index=True)
configuration_lines = "\n".join(
    f"{index}. `{json.dumps(parameters, sort_keys=True)}`"
    for index, parameters in enumerate(candidate_parameters, start=1)
)
report = f"""# LightGBM Temporal-CV Tuning

## Methodology and leakage protections

Three expanding folds are built inside TRAIN. Boundaries align to complete timestamps and
each fit period strictly precedes evaluation. Frequency maps, medians, scaling, historical
amount state, and graph state are rebuilt using each fold's fit history only. Fold evaluation
drives early stopping with a 50-round patience. External VALIDATION is not transformed or
scored until both configurations are selected. The final holdout is untouched.

{fold_markdown(fold_design)}

## Configurations tested

Eight curated parameter configurations were applied to both feature candidates (16 CV runs).

{configuration_lines}

## TRAIN-only temporal CV results

{cv_markdown(combined_cv)}

- Candidate A best configuration: **{int(best_a_row["Configuration"])}**
- Candidate A CV parameters: `{json.dumps(best_a.parameters, sort_keys=True)}`
- Candidate A fold iterations: **{best_a.best_iterations}**; full-TRAIN refit uses
  **{simple_result["Parameters"]["n_estimators"]}** estimators.
- Candidate B best configuration: **{int(best_b_row["Configuration"])}**
- Candidate B CV parameters: `{json.dumps(best_b.parameters, sort_keys=True)}`
- Candidate B fold iterations: **{best_b.best_iterations}**; full-TRAIN refit uses
  **{graph_result["Parameters"]["n_estimators"]}** estimators.

## One-time external validation results

{final_markdown(final_results)}

Previous untuned results were `0.593938` for Candidate A and `0.594611` for Candidate B.
The tuned winner changes the previous overall best by **{improvement_over_previous:+.6f}**.

## Simple versus graph trade-off

{tradeoff}

Graph feature generation previously required 60.8 seconds and sparse relationship state;
Candidate A avoids that operational dependency.

## MLflow

- Experiment: `fraud-detection-lightgbm-tuning`
- Candidate A winning CV run: `{best_a_row["MLflow Run ID"]}`
- Candidate B winning CV run: `{best_b_row["MLflow Run ID"]}`
- Candidate A final-refit run: `{simple_result["MLflow Run ID"]}`
- Candidate B final-refit run: `{graph_result["MLflow Run ID"]}`

## Runtime and memory

- Candidate A CV workflow: **{preparation_seconds["candidate_a"]:.1f} seconds**
- Candidate B CV workflow: **{preparation_seconds["candidate_b_graph"]:.1f} seconds**
- Total experiment before report write: **{time.perf_counter() - notebook_started:.1f} seconds**
- Largest simultaneously prepared matrix pair: **{maximum_matrix_mb:.1f} MiB**
- Matrices are prepared one fold at a time and scaling statistics are learned in batches.

No calibration, threshold optimization, SHAP, rules, registry promotion, or holdout evaluation
is performed.
"""
report_path = PROJECT_ROOT / "reports" / "lightgbm_tuning_summary.md"
report_path.write_text(report, encoding="utf-8")
print(f"Winner: {winner['Feature Set']} PR-AUC={winner['PR-AUC']:.6f}")
print(f"Improvement over 0.594611: {improvement_over_previous:+.6f}")
print(tradeoff)
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")

Winner: candidate_a PR-AUC=0.593452
Improvement over 0.594611: -0.001159
Graph features do not improve tuned validation PR-AUC; the simpler 443-feature model is preferable on both ranking performance and operational complexity.
Report: reports\lightgbm_tuning_summary.md


## Scope boundary

This notebook stops after LightGBM temporal-CV tuning and one-time external validation comparison. It does not perform calibration, threshold optimization, business rules, SHAP, holdout evaluation, retraining, serving, or deployment.